# Femora — Breast Ultrasound Classifier (ResNet50)

**Datasets (combined)**
- **BUSI**: 780 images from 600 women, Baheya Hospital, Cairo (Al-Dhabyani et al., 2020). Labels: normal / benign / malignant.
- **BrEaST-Lesions-USG**: 256 scans from 256 patients, every label confirmed by biopsy or follow-up (Pawłowska et al., 2024,
  The Cancer Imaging Archive, CC BY 4.0). A different hospital and different scanners, so it tests whether the model
  generalises beyond one clinic.

**Design decisions**
- **ResNet50 pretrained on ImageNet**, fine-tuned in two phases (classifier head first, then the whole network) with a
  class-weighted loss, because malignant and normal scans are the minority.
- **Grayscale, padded to a square**: scanners tint images differently and colour carries no information in B-mode
  ultrasound. Padding (instead of stretching) keeps lesion shape, which matters because irregular shape is a malignancy sign.
- **Near-duplicate images are grouped** by perceptual hash (BUSI contains several), so a copy of a test image can never be
  in the training set.
- **Two training recipes are compared by 5-fold cross-validation**: the baseline, and a recipe that weights both hospitals
  equally and adds scanner-style augmentation. The winner is chosen by a rule fixed in advance: the average malignant
  ROC-AUC across the two hospitals, on out-of-fold predictions. The test set is not used for any decision.
- **Screening threshold**: a screening tool should rarely miss cancer, so the "suspicious" threshold is tuned for ≥ 90%
  malignant sensitivity on ~870 out-of-fold predictions (a small validation split gives a noisy threshold).
- **Explainability**: Class Activation Maps (exact for a global-average-pool + linear head), checked against the
  radiologists' lesion masks.
- **Safety**: uploads that are not breast ultrasounds are rejected by a colour check and an "is this an ultrasound?"
  gate, evaluated on image types it never saw during training.
- **Every reported test number is computed with the exported ONNX model**, which is exactly what the Femora backend runs.

Output: normal / benign / malignant probabilities plus a heatmap. This is awareness only, not a diagnosis.

In [ ]:
%pip install -q onnx onnxruntime onnxscript

In [ ]:
import copy, glob, json, os, random, shutil, urllib.request, zipfile, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from PIL import Image, ImageOps
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, recall_score, confusion_matrix,
                             classification_report, roc_curve)
warnings.filterwarnings("ignore")

# Paths can be overridden so the notebook can be smoke-tested locally before a Kaggle run
INPUT = os.environ.get("FEMORA_INPUT", "/kaggle/input")
WORK = os.environ.get("FEMORA_WORK", "/kaggle/working")
TMP = os.environ.get("FEMORA_TMP", "/tmp/femora")      # downloads go here so they don't bloat the notebook output
SMOKE = os.environ.get("FEMORA_SMOKE") == "1"          # tiny run: a few images, one epoch
OUT = f"{WORK}/model"
for d in (OUT, TMP):
    os.makedirs(d, exist_ok=True)

SEED = 42
CLASSES = ["normal", "benign", "malignant"]
SOURCES = ["BUSI", "BrEaST"]
MAL = CLASSES.index("malignant")
IMG = 224
MEAN = np.array([0.485, 0.456, 0.406], np.float32)[:, None, None]   # ImageNet statistics (pretrained backbone)
STD = np.array([0.229, 0.224, 0.225], np.float32)[:, None, None]
COLOUR_LIMIT = 0.10       # max share of clearly coloured pixels in an upload
BATCH = 8 if SMOKE else 32
EPOCHS_HEAD, EPOCHS_FT, CV_FOLDS = (1, 1, 2) if SMOKE else (4, 26, 5)
NUM_WORKERS = 0 if os.name == "nt" else 4

def seed_everything(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def pick_device():
    if torch.cuda.is_available():
        try:  # fail over to CPU if this GPU architecture isn't in the installed torch build
            (torch.ones(2, 2, device="cuda") @ torch.ones(2, 2, device="cuda")).sum().item()
            return torch.device("cuda")
        except RuntimeError as e:
            print("GPU not usable:", e)
    return torch.device("cpu")

seed_everything()
DEVICE = pick_device()
print(DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "", "torch", torch.__version__)

# ---- Preprocessing shared with backend/app.py (keep the two in sync)
def pad_square(img, fill=0):
    w, h = img.size
    s = max(w, h)
    canvas = Image.new(img.mode, (s, s), fill)
    canvas.paste(img, ((s - w) // 2, (s - h) // 2))
    return canvas

def to_gray224(img):
    # any upload -> 224x224 uint8 grayscale, padded (not stretched) to a square
    return np.asarray(pad_square(ImageOps.exif_transpose(img).convert("L")).resize((IMG, IMG), Image.BILINEAR))

def normalize(gray224):
    x = gray224.astype(np.float32) / 255.0
    return ((np.repeat(x[None], 3, 0) - MEAN) / STD).astype(np.float32)

def colour_fraction(img):
    # share of clearly coloured pixels: B-mode ultrasound is grey (sometimes tinted), photos and Doppler are not
    a = np.asarray(img.convert("RGB").resize((128, 128)), np.int16)
    diff = np.maximum.reduce([abs(a[..., 0] - a[..., 1]), abs(a[..., 1] - a[..., 2]), abs(a[..., 0] - a[..., 2])])
    return float((diff > 30).mean())

def softmax(z):
    e = np.exp(z - z.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)

def download(url, dest):
    if not os.path.exists(dest):
        request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(request) as r, open(dest, "wb") as f:
            shutil.copyfileobj(r, f)
    return dest

## 1. Load both datasets

In [ ]:
# BUSI (Kaggle): <class>/<name>.png, lesion masks are <name>_mask.png, <name>_mask_1.png, ...
busi_root = glob.glob(f"{INPUT}/**/Dataset_BUSI_with_GT", recursive=True)[0]
rows = []
for label in CLASSES:
    for path in sorted(glob.glob(f"{busi_root}/{label}/*.png")):
        if "_mask" not in os.path.basename(path):
            masks = sorted(glob.glob(glob.escape(path[:-4]) + "_mask*.png"))
            rows.append({"path": path, "source": "BUSI", "label": label, "masks": masks})

# BrEaST-Lesions-USG (TCIA, CC BY 4.0): one scan per patient, labels confirmed by biopsy or follow-up
TCIA = "https://www.cancerimagingarchive.net/wp-content/uploads/"
zip_path = download(TCIA + "BrEaST-Lesions_USG-images_and_masks-Dec-15-2023.zip", f"{TMP}/breast_usg.zip")
xlsx_path = download(TCIA + "BrEaST-Lesions-USG-clinical-data-Dec-15-2023.xlsx", f"{TMP}/breast_usg.xlsx")
if not glob.glob(f"{TMP}/breast_usg/**/case001.png", recursive=True):
    zipfile.ZipFile(zip_path).extractall(f"{TMP}/breast_usg")
img_dir = os.path.dirname(glob.glob(f"{TMP}/breast_usg/**/case001.png", recursive=True)[0])
clinical = pd.read_excel(xlsx_path)
for r in clinical.itertuples():
    masks = [f"{img_dir}/{m}" for m in r.Mask_tumor_filename.split("&")] if isinstance(r.Mask_tumor_filename, str) else []
    rows.append({"path": f"{img_dir}/{r.Image_filename}", "source": "BrEaST", "label": r.Classification, "masks": masks})

df = pd.DataFrame(rows)
assert set(df["label"]) <= set(CLASSES), set(df["label"])
if SMOKE:
    df = df.groupby(["source", "label"], group_keys=False).head(12).reset_index(drop=True)
print(pd.crosstab(df["source"], df["label"], margins=True))

fig, axes = plt.subplots(2, 6, figsize=(16, 5.5))
for ax in axes.flat:
    ax.axis("off")
for ax, r in zip(axes.T.flat, df.groupby(["source", "label"]).head(2).itertuples()):
    ax.imshow(Image.open(r.path).convert("L"), cmap="gray")
    ax.set_title(f"{r.source} · {r.label}", fontsize=10)
plt.tight_layout(); plt.show()

## 2. Remove near-duplicates, then split by duplicate group

Each image gets a 256-bit difference hash. Images whose hashes differ in ≤ 12 bits (≈ 5%) are treated as copies of the same
scan and kept in the same split. Copies with *conflicting* labels are dropped, because we can't tell which label is correct.
The split is stratified by source × class: 5/7 train, 1/7 validation, 1/7 test.

In [ ]:
def dhash(path, size=16):
    g = np.asarray(Image.open(path).convert("L").resize((size + 1, size), Image.BILINEAR), dtype=np.int16)
    return (g[:, 1:] > g[:, :-1]).flatten()

h = np.stack([dhash(p) for p in df["path"]]).astype(np.float32) * 2 - 1
hamming = (h.shape[1] - h @ h.T) / 2
parent = list(range(len(df)))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i
for i, j in zip(*np.where(np.triu(hamming <= 12, k=1))):
    parent[find(i)] = find(j)
df["group"] = [find(i) for i in range(len(df))]

dups = df[df.duplicated("group", keep=False)]
n_labels = dups.groupby("group")["label"].nunique()
conflicting = n_labels[n_labels > 1].index
print(f"{len(dups)} images fall into {dups['group'].nunique()} near-duplicate groups; "
      f"{len(conflicting)} groups have conflicting labels and are dropped "
      f"({df['group'].isin(conflicting).sum()} images)")

examples = dups.groupby("group").head(2).head(8)
if len(examples):
    fig, axes = plt.subplots(1, len(examples), figsize=(2.4 * len(examples), 2.8))
    for ax, r in zip(np.atleast_1d(axes), examples.itertuples()):
        ax.imshow(Image.open(r.path).convert("L"), cmap="gray"); ax.axis("off")
        ax.set_title(f"group {r.group}\n{r.label}", fontsize=8)
    plt.suptitle("Examples of detected near-duplicates (pairs)"); plt.tight_layout(); plt.show()

dedup_stats = {"images_in_duplicate_groups": int(len(dups)), "duplicate_groups": int(dups["group"].nunique()),
               "dropped_conflicting_images": int(df["group"].isin(conflicting).sum())}
df = df[~df["group"].isin(conflicting)].reset_index(drop=True)
df["strata"] = df["source"] + "_" + df["label"]

folds = np.zeros(len(df), int)
for k, (_, test_part) in enumerate(StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
                                   .split(df, df["strata"], df["group"])):
    folds[test_part] = k
df["split"] = np.select([folds == 0, folds == 1], ["test", "val"], "train")
idx = {s: np.where(df["split"] == s)[0] for s in ["train", "val", "test"]}
print(pd.crosstab([df["source"], df["label"]], df["split"], margins=True))

## 3. Preprocessing, augmentation and the two training recipes

- **baseline**: mild augmentation; classes weighted by inverse frequency.
- **source-balanced + scanner augmentation**: BUSI makes up ~75% of the images, so a model can score well by learning
  what BUSI's scanners look like. This recipe gives each hospital equal total weight and adds augmentation that mimics
  differences between machines (zoom, blur, sharpness, gain and contrast).

In [ ]:
labels = df["label"].map(CLASSES.index).values
sources = df["source"].values
gray224 = np.stack([to_gray224(Image.open(p)) for p in df["path"]])   # exactly what the backend feeds the model
train_imgs = [Image.fromarray(np.asarray(pad_square(ImageOps.exif_transpose(Image.open(p)).convert("L"))
                                         .resize((288, 288), Image.BILINEAR))) for p in df["path"]]

AUGMENT = {
    "standard": transforms.Compose([
        transforms.RandomResizedCrop(IMG, scale=(0.7, 1.0), ratio=(0.85, 1.18)),
        transforms.RandomHorizontalFlip(),   # no vertical flip: the transducer is always at the top of the image
        transforms.RandomApply([transforms.RandomRotation(10)], p=0.5),
        transforms.ColorJitter(brightness=0.25, contrast=0.25),
        transforms.PILToTensor(),
    ]),
    "scanner": transforms.Compose([
        transforms.RandomResizedCrop(IMG, scale=(0.55, 1.0), ratio=(0.8, 1.25)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomApply([transforms.RandomRotation(12)], p=0.5),
        transforms.ColorJitter(brightness=0.4, contrast=0.4),
        transforms.RandomApply([transforms.GaussianBlur(5, sigma=(0.1, 1.5))], p=0.3),
        transforms.RandomAdjustSharpness(2.0, p=0.3),
        transforms.PILToTensor(),
    ]),
}
RECIPES = {
    "baseline": {"augment": "standard", "balance_sources": False},
    "source-balanced + scanner augmentation": {"augment": "scanner", "balance_sources": True},
}

def sample_weights(ids, balance_sources):
    # inverse class frequency, optionally times inverse source frequency so both hospitals count equally
    y, src = labels[ids], sources[ids]
    w = (len(ids) / (len(CLASSES) * np.maximum(np.bincount(y, minlength=len(CLASSES)), 1)))[y]
    if balance_sources:
        names, counts = np.unique(src, return_counts=True)
        per_source = dict(zip(names, len(ids) / (len(names) * counts)))
        w = w * np.array([per_source[s] for s in src])
    out = np.zeros(len(df), np.float32)
    out[ids] = w / w.mean()
    return out

class UltrasoundDS(Dataset):
    def __init__(self, ids, augment=None):
        self.ids, self.augment = np.asarray(ids), augment
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        j = self.ids[i]
        g = AUGMENT[self.augment](train_imgs[j])[0].numpy() if self.augment else gray224[j]
        return torch.from_numpy(normalize(g)), int(labels[j]), int(j)

def loader(ids, augment=None):
    return DataLoader(UltrasoundDS(ids, augment), batch_size=BATCH, shuffle=augment is not None,
                      drop_last=augment is not None, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])   # -> B x 2048 x 7 x 7 feature maps
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Linear(2048, len(CLASSES))
    def forward(self, x):
        emb = self.backbone(x).mean((2, 3))   # global average pooling
        return self.fc(self.drop(emb)), emb

def predict_logits(net, ids, amp=True):
    net.eval()
    out = []
    with torch.no_grad():
        for x, *_ in loader(ids):
            with torch.autocast(DEVICE.type, enabled=amp and DEVICE.type == "cuda"):
                out.append(net(x.to(DEVICE))[0].float().cpu())
    return torch.cat(out).numpy()

def summarize(y, p, pred=None):
    pred = p.argmax(1) if pred is None else pred
    return {"accuracy": round(float(accuracy_score(y, pred)), 4),
            "macro_f1": round(float(f1_score(y, pred, average="macro")), 4),
            "malignant_sensitivity": round(float(recall_score(y == MAL, pred == MAL)), 4),
            "malignant_specificity": round(float(recall_score(y != MAL, pred != MAL)), 4),
            "malignant_auc": round(float(roc_auc_score(y == MAL, p[:, MAL])), 4)}

def summarize_by_source(ids, p, pred=None):
    out = {}
    for src in SOURCES:
        m = sources[ids] == src
        try:
            out[src] = {"images": int(m.sum()), **summarize(labels[ids][m], p[m], None if pred is None else pred[m])}
        except ValueError as e:   # e.g. a class missing from a tiny smoke-test split
            print(src, e)
    return out

fig, axes = plt.subplots(2, 6, figsize=(15, 5.6))
for row, name in zip(axes, AUGMENT):
    for ax in row:
        ax.imshow(AUGMENT[name](train_imgs[idx["train"][0]])[0], cmap="gray"); ax.axis("off")
    row[0].set_title(f"{name} augmentation", loc="left", fontsize=10)
plt.tight_layout(); plt.show()

## 4. Baseline: frozen ImageNet ResNet50 + logistic regression

A *linear probe* uses ResNet50 exactly as trained on everyday photos, with no fine-tuning. It shows how much the
fine-tuning in the next sections actually adds.

In [ ]:
@torch.no_grad()
def embed(net, ids):
    net.eval()
    return torch.cat([net(x.to(DEVICE))[1].float().cpu() for x, *_ in loader(ids)]).numpy()

imagenet_net = Net().to(DEVICE)
probe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, C=0.05, class_weight="balanced"))
probe.fit(embed(imagenet_net, idx["train"]), labels[idx["train"]])
p_probe_test = probe.predict_proba(embed(imagenet_net, idx["test"]))
baseline = summarize(labels[idx["test"]], p_probe_test)
del imagenet_net
print(json.dumps(baseline, indent=2))

## 5. Choose the training recipe by 5-fold cross-validation (development set only)

Phase 1 trains only the new classifier head (backbone frozen). Phase 2 unfreezes the whole network, with a 5× lower
learning rate for the pretrained layers and cosine decay. Folds use the last epoch (no checkpoint picking), so the
estimate is not optimistically biased. The test set is not touched.

**Selection rule (fixed before running):** the recipe with the higher average of the out-of-fold malignant ROC-AUC on
BUSI and on BrEaST, so doing well on one hospital cannot hide doing badly on the other.

In [ ]:
def train_model(train_ids, val_ids, recipe, select_best=True, verbose=False):
    seed_everything()
    net = Net().to(DEVICE)
    weights = torch.tensor(sample_weights(train_ids, recipe["balance_sources"]), device=DEVICE)
    criterion = nn.CrossEntropyLoss(reduction="none", label_smoothing=0.05)
    train_dl = loader(train_ids, recipe["augment"])
    scaler = torch.amp.GradScaler(enabled=DEVICE.type == "cuda")

    for p in net.backbone.parameters():
        p.requires_grad = False
    opt, sched = torch.optim.AdamW(net.fc.parameters(), lr=1e-3, weight_decay=1e-4), None
    best, history = (-1.0, None, 0), []
    for epoch in range(EPOCHS_HEAD + EPOCHS_FT):
        if epoch == EPOCHS_HEAD:   # phase 2: fine-tune everything
            for p in net.backbone.parameters():
                p.requires_grad = True
            opt = torch.optim.AdamW([{"params": net.backbone.parameters(), "lr": 1e-4},
                                     {"params": net.fc.parameters(), "lr": 5e-4}], weight_decay=1e-4)
            sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_FT * len(train_dl))
        net.train()
        loss_sum, seen = 0.0, 0
        for x, y, j in train_dl:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
                loss = (criterion(net(x)[0], y) * weights[j.to(DEVICE)]).mean()
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            if sched:
                sched.step()
            loss_sum, seen = loss_sum + loss.item() * len(y), seen + len(y)
        val_f1 = f1_score(labels[val_ids], predict_logits(net, val_ids).argmax(1), average="macro")
        history.append({"epoch": epoch + 1, "train_loss": loss_sum / max(seen, 1), "val_macro_f1": val_f1})
        if val_f1 > best[0]:
            best = (val_f1, copy.deepcopy(net.state_dict()), epoch + 1)
        if verbose:
            print(f"epoch {epoch + 1:2d}  loss {history[-1]['train_loss']:.3f}  val macro-F1 {val_f1:.3f}")
    if select_best:
        net.load_state_dict(best[1])
    return net, pd.DataFrame(history), best[2]

dev = np.concatenate([idx["train"], idx["val"]])
cv_splits = list(StratifiedGroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
                 .split(dev, df["strata"].values[dev], df["group"].values[dev]))
oof_logits, cv_rows = {}, []
for name, recipe in RECIPES.items():
    oof_logits[name] = np.zeros((len(df), len(CLASSES)), np.float32)
    for k, (tr, va) in enumerate(cv_splits):
        fold_net, _, _ = train_model(dev[tr], dev[va], recipe, select_best=False)
        logits = predict_logits(fold_net, dev[va], amp=False)
        oof_logits[name][dev[va]] = logits
        cv_rows.append({"recipe": name, "fold": k + 1, **summarize(labels[dev[va]], softmax(logits))})
        print(cv_rows[-1])
        del fold_net
        torch.cuda.empty_cache()
cv_table = pd.DataFrame(cv_rows)

recipe_rows = []
for name in RECIPES:
    p_oof = softmax(oof_logits[name][dev])
    by_source = summarize_by_source(dev, p_oof)
    folds_of = cv_table[cv_table.recipe == name]
    recipe_rows.append({
        "recipe": name,
        "selection_score": round(float(np.mean([by_source[s]["malignant_auc"] for s in by_source])), 4),
        **{f"cv_{m}": f"{folds_of[m].mean():.3f} ± {folds_of[m].std():.3f}"
           for m in ["accuracy", "macro_f1", "malignant_sensitivity", "malignant_auc"]},
        **{f"{s}_{m}": by_source[s][m] for s in by_source for m in ["accuracy", "malignant_auc"]},
    })
recipe_table = pd.DataFrame(recipe_rows)
BEST_RECIPE = recipe_table.loc[recipe_table["selection_score"].idxmax(), "recipe"]
display(recipe_table)
print("Selected recipe:", BEST_RECIPE)
cv_summary = recipe_table.set_index("recipe").loc[BEST_RECIPE].to_dict()

## 6. Final model (selected recipe, train split, best epoch picked on the validation split)

In [ ]:
net, history, best_epoch = train_model(idx["train"], idx["val"], RECIPES[BEST_RECIPE], verbose=True)
p_torch_test = softmax(predict_logits(net, idx["test"], amp=False))   # full-precision reference for the parity check

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(history["epoch"], history["train_loss"], color="#C2185B", label="train loss")
ax2 = ax1.twinx()
ax2.plot(history["epoch"], history["val_macro_f1"], color="#6C2D7E", label="val macro-F1")
ax1.axvline(best_epoch, ls="--", color="grey"); ax1.axvline(EPOCHS_HEAD + 0.5, ls=":", color="lightgrey")
ax1.set(xlabel="epoch", ylabel="train loss"); ax2.set_ylabel("val macro-F1")
fig.legend(loc="upper center", ncol=2); plt.title(f"Training (best epoch {best_epoch})", pad=24)
plt.tight_layout(); plt.savefig(f"{OUT}/breast_training_curve.png", dpi=150); plt.show()

## 7. Export to ONNX and verify parity

The exported graph returns three outputs: **logits** (the prediction), the **embedding** (used by the ultrasound gate)
and **class activation maps** (the heatmap). With a global-average-pool + linear head, CAM is exact: it is the
classifier's weights applied to every spatial location of the last feature map.

The fp32 ResNet50 is 94 MB, close to GitHub's 100 MB file limit. Conv/linear weights are therefore **stored** in fp16 and
cast back to fp32 when the model loads. All computation stays in fp32.

In [ ]:
import onnx, onnxruntime as ort
from onnx import helper, numpy_helper, TensorProto

class ExportNet(nn.Module):
    def __init__(self, net):
        super().__init__()
        self.net = net
    def forward(self, x):
        fmap = self.net.backbone(x)
        emb = fmap.mean((2, 3))
        cam = F.conv2d(fmap, self.net.fc.weight[:, :, None, None])   # class activation maps, B x 3 x 7 x 7
        return self.net.fc(emb), emb, cam

net = net.float().cpu().eval()
fp32_path = f"{TMP}/breast_resnet50_fp32.onnx"
export_args = dict(input_names=["image"], output_names=["logits", "embedding", "cam"], opset_version=17,
                   dynamic_axes={n: {0: "batch"} for n in ["image", "logits", "embedding", "cam"]})
dummy = torch.from_numpy(normalize(gray224[0]))[None]
try:
    torch.onnx.export(ExportNet(net).eval(), dummy, fp32_path, dynamo=False, **export_args)
except TypeError:   # torch < 2.5 has no `dynamo` argument
    torch.onnx.export(ExportNet(net).eval(), dummy, fp32_path, **export_args)

model = onnx.load(fp32_path)
graph = model.graph
initializers, casts = [], []
for init in graph.initializer:
    w = numpy_helper.to_array(init)
    if w.dtype == np.float32 and w.ndim >= 2:   # conv / linear weights; BatchNorm statistics stay fp32
        initializers.append(numpy_helper.from_array(w.astype(np.float16), init.name + "_fp16"))
        casts.append(helper.make_node("Cast", [init.name + "_fp16"], [init.name], to=TensorProto.FLOAT))
    else:
        initializers.append(numpy_helper.from_array(w, init.name))
nodes = casts + [copy.deepcopy(n) for n in graph.node]
graph.ClearField("initializer"); graph.initializer.extend(initializers)
graph.ClearField("node"); graph.node.extend(nodes)
onnx.checker.check_model(model)
ONNX_PATH = f"{OUT}/breast_resnet50.onnx"
onnx.save(model, ONNX_PATH)
print(f"fp32 {os.path.getsize(fp32_path) / 1e6:.1f} MB -> deployed {os.path.getsize(ONNX_PATH) / 1e6:.1f} MB")

def run_session(sess, grays, batch=32):
    outs = [sess.run(None, {"image": np.stack([normalize(g) for g in grays[i:i + batch]])})
            for i in range(0, len(grays), batch)]
    return [np.concatenate(o) for o in zip(*outs)]

session = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])
def run_onnx(grays):
    return run_session(session, grays)

onnx_out = {s: run_onnx(gray224[idx[s]]) for s in ["train", "val", "test"]}   # (logits, embedding, cam) per split
# Two separate checks: the export itself must be exact; fp16 weight storage only adds rounding noise.
# (All metrics below are computed with the deployed model, so that noise is already reflected in them.)
fp32_session = ort.InferenceSession(fp32_path, providers=["CPUExecutionProvider"])
p_onnx_test = softmax(onnx_out["test"][0])
parity_fp32 = float(np.abs(softmax(run_session(fp32_session, gray224[idx["test"]])[0]) - p_torch_test).max())
parity = float(np.abs(p_onnx_test - p_torch_test).max())
flipped = int((p_onnx_test.argmax(1) != p_torch_test.argmax(1)).sum())
print(f"max |P_onnx - P_torch| on the test set: fp32 export {parity_fp32:.5f}, deployed (fp16 weights) {parity:.5f}; "
      f"{flipped} of {len(p_torch_test)} test predictions change class")
assert parity_fp32 < 1e-3, "ONNX export disagrees with the trained network"
assert parity < 0.05, "fp16 weight storage changes predictions too much"

## 8. Calibration and the screening threshold (out-of-fold predictions)

- **Temperature scaling** makes the displayed confidence honest: when the app says "85% confident", it should be right
  about 85% of the time.
- **Screening rule**: a scan is flagged *suspicious* when P(malignant) ≥ threshold, even if another class is more likely.
  The threshold is the highest value that still catches ≥ 90% of malignant scans.

Both are fitted on the selected recipe's out-of-fold predictions for the whole development set (~870 scans, ~260
malignant). A threshold fitted on the small validation split alone (~30 malignant scans) is too noisy.

In [ ]:
def fit_temperature(logits, y):
    log_t = torch.zeros(1, requires_grad=True)
    z, target = torch.tensor(logits), torch.tensor(y)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=200)
    def closure():
        opt.zero_grad()
        loss = F.cross_entropy(z / log_t.exp(), target)
        loss.backward()
        return loss
    opt.step(closure)
    return float(log_t.exp())

def ece(p, y, bins=10):
    # expected calibration error: average gap between confidence and accuracy, weighted by bin size
    conf, correct = p.max(1), p.argmax(1) == y
    edges = np.linspace(0, 1, bins + 1)
    total = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            total += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(total)

def decide(p, t_mal):
    pred = p.argmax(1)
    pred[p[:, MAL] >= t_mal] = MAL
    return pred

z_oof, y_dev = oof_logits[BEST_RECIPE][dev], labels[dev]
TEMPERATURE = fit_temperature(z_oof, y_dev)
p_oof = softmax(z_oof / TEMPERATURE)
ok = [t for t in np.round(np.arange(0.50, 0.04, -0.01), 2)
      if recall_score(y_dev == MAL, decide(p_oof, t) == MAL) >= 0.90]
MAL_THRESHOLD = float(ok[0]) if ok else 0.10
y_val, y_test = labels[idx["val"]], labels[idx["test"]]
calibration = {"temperature": round(TEMPERATURE, 4), "fitted_on": f"{len(dev)} out-of-fold predictions",
               "oof_ece_before": round(ece(softmax(z_oof), y_dev), 4), "oof_ece_after": round(ece(p_oof, y_dev), 4),
               "val_ece_final_model": round(ece(softmax(onnx_out["val"][0] / TEMPERATURE), y_val), 4)}
print(calibration, "| malignant threshold:", MAL_THRESHOLD)

## 9. Held-out test set (computed with the exported ONNX model)

In [ ]:
p_test = softmax(onnx_out["test"][0] / TEMPERATURE)
pred_test = decide(p_test, MAL_THRESHOLD)
test_metrics = {"test_size": int(len(y_test)), **summarize(y_test, p_test, pred_test),
                "macro_auc_ovr": round(float(roc_auc_score(y_test, p_test, multi_class="ovr")), 4),
                "ece": round(ece(p_test, y_test), 4)}
calibration["test_ece_before"] = round(ece(softmax(onnx_out["test"][0]), y_test), 4)
calibration["test_ece_after"] = test_metrics["ece"]
argmax_metrics = summarize(y_test, p_test)
print(json.dumps(test_metrics, indent=2))
print(classification_report(y_test, pred_test, target_names=CLASSES))

per_source = summarize_by_source(idx["test"], p_test, pred_test)
print(pd.DataFrame(per_source).T)

comparison = pd.DataFrame([
    {"model": "Linear probe (frozen ImageNet ResNet50)", **baseline},
    {"model": "Fine-tuned ResNet50, argmax", **argmax_metrics},
    {"model": f"Fine-tuned ResNet50, screening rule (P(mal) >= {MAL_THRESHOLD})", **summarize(y_test, p_test, pred_test)},
])
display(comparison)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
sns.heatmap(confusion_matrix(y_test, pred_test, labels=range(len(CLASSES))), annot=True, fmt="d", cmap="RdPu",
            ax=axes[0], xticklabels=CLASSES, yticklabels=CLASSES)
axes[0].set(title="Confusion matrix (test set, screening rule)", xlabel="Predicted", ylabel="Actual")
for src, colour in [("BUSI", "#C2185B"), ("BrEaST", "#6C2D7E")]:
    m = sources[idx["test"]] == src
    fpr, tpr, _ = roc_curve(y_test[m] == MAL, p_test[m, MAL])
    axes[1].plot(fpr, tpr, color=colour, label=f"{src} (AUC {per_source[src]['malignant_auc']:.2f})")
axes[1].plot([0, 1], [0, 1], "--", color="grey")
axes[1].set(xlabel="False positive rate", ylabel="True positive rate", title="Malignant vs rest, by hospital")
axes[1].legend(loc="lower right")
plt.tight_layout(); plt.savefig(f"{OUT}/breast_evaluation.png", dpi=150); plt.show()

## 10. Explainability: do the heatmaps land on the lesion?

For every test scan with a lesion, the activation map of its true class is upsampled to the image. Two scores:
- **Pointing game**: is the hottest pixel inside the radiologist's lesion mask?
- **IoU**: overlap between the hot region (≥ 50% of the maximum) and the mask.

In [ ]:
def upsample_cam(cam7):
    c = np.maximum(cam7, 0)
    c = c / c.max() if c.max() > 0 else c
    return np.asarray(Image.fromarray((c * 255).astype(np.uint8)).resize((IMG, IMG), Image.BILINEAR)) / 255.0

def mask224(paths):
    m = np.zeros((IMG, IMG), bool)
    for p in paths:
        m |= np.asarray(pad_square(Image.open(p).convert("L")).resize((IMG, IMG), Image.NEAREST)) > 127
    return m

cam_test = onnx_out["test"][2]
hits, ious, chance, shown = [], [], [], []
for k, j in enumerate(idx["test"]):
    if labels[j] == CLASSES.index("normal") or not df.at[j, "masks"]:
        continue
    mask = mask224(df.at[j, "masks"])
    if not mask.any():
        continue
    heat = upsample_cam(cam_test[k, labels[j]])
    hits.append(bool(mask[np.unravel_index(heat.argmax(), heat.shape)]))
    hot = heat >= 0.5
    ious.append((hot & mask).sum() / (hot | mask).sum())
    chance.append(mask.sum() / (gray224[j] > 10).sum())   # hit rate of a random point on the scan
    if len(shown) < 8 and (len(shown) < 4 or labels[j] == MAL):
        shown.append((j, heat, mask))
cam_eval = {"lesion_images": len(hits), "pointing_game": round(float(np.mean(hits)), 4),
            "mean_iou": round(float(np.mean(ious)), 4), "pointing_game_chance": round(float(np.mean(chance)), 4)}
print(cam_eval)

fig, axes = plt.subplots(2, 4, figsize=(14, 7.5))
for ax in axes.flat:
    ax.axis("off")
for ax, (j, heat, mask) in zip(axes.flat, shown):
    ax.imshow(gray224[j], cmap="gray"); ax.imshow(heat, cmap="jet", alpha=0.35)
    ax.contour(mask, levels=[0.5], colors="white", linewidths=1.2)
    ax.set_title(f"{df.at[j, 'label']} ({df.at[j, 'source']})", fontsize=10)
plt.suptitle("Class activation maps (colour) vs radiologist lesion masks (white outline)")
plt.tight_layout(); plt.savefig(f"{OUT}/breast_cam_examples.png", dpi=130); plt.show()

## 11. Rejecting uploads that aren't breast ultrasounds

People will upload the wrong image: a selfie, a photo of a report, an X-ray. The classifier would still output
normal/benign/malignant for any of these, so the backend runs two checks first:
1. **Colour check**: B-mode ultrasound is grey. Images where more than 10% of pixels are clearly coloured are rejected.
2. **Ultrasound gate**: a logistic regression on the network's embedding, trained to separate the training ultrasounds
   from *grayscale* photos (4 categories) and chest X-rays. Its threshold lets 99% of validation scans through.

To check that the gate generalises, it is tested on held-out ultrasounds and on images it never saw: other photo
categories (in colour and in grayscale), held-out chest X-rays, and brain MRIs, an image type absent from its training.
(An unsupervised feature-distance check was tried first and rejected almost nothing: photos and X-rays landed at the
same distances as real scans.)

In [ ]:
def find_images(pattern):
    paths = glob.glob(f"{INPUT}/**/{pattern}", recursive=True)
    return sorted(p for p in paths if "__MACOSX" not in p and not os.path.basename(p).startswith("._")
                  and p.lower().endswith((".jpg", ".jpeg", ".png")))

def load_images(paths):
    images = []
    for p in paths:
        try:
            im = Image.open(p)
            im.load()
            images.append(im)
        except Exception:
            pass
    return images

rng = np.random.default_rng(SEED)
def sample(paths, n):
    return list(rng.choice(paths, min(n, len(paths)), replace=False)) if paths else []

GATE_PHOTO_CLASSES = ["airplane", "car", "cat", "dog"]   # the other natural-image classes are held out
natural = find_images("natural_images/*/*.jpg")
photos_train = load_images(sample([p for p in natural if Path(p).parent.name in GATE_PHOTO_CLASSES], 600))
photos_heldout = load_images(sample([p for p in natural if Path(p).parent.name not in GATE_PHOTO_CLASSES], 300))
xray_train = load_images(sample(find_images("chest_xray/train/*/*.jpeg"), 400))
xray_heldout = load_images(sample(find_images("chest_xray/test/*/*.jpeg"), 300))
mri = load_images(sample(find_images("brain_tumor_dataset/*/*"), 250))
print({"photos (train)": len(photos_train), "photos (held out)": len(photos_heldout), "x-rays (train)": len(xray_train),
       "x-rays (held out)": len(xray_heldout), "brain MRI (never seen)": len(mri)})

def embeddings(images):
    return run_onnx(np.stack([to_gray224(im) for im in images]))[1] if images else np.zeros((0, 2048), np.float32)

negatives = np.concatenate([embeddings(photos_train), embeddings(xray_train)])
X_gate = np.concatenate([onnx_out["train"][1], negatives])
y_gate = np.r_[np.ones(len(idx["train"])), np.zeros(len(negatives))]
gate = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=5000, class_weight="balanced")).fit(X_gate, y_gate)
gate_score = lambda emb: gate.predict_proba(emb)[:, 1] if len(emb) else np.zeros(0)
GATE_THRESHOLD = float(min(np.percentile(gate_score(onnx_out["val"][1]), 1), 0.5))

test_colour = np.array([colour_fraction(Image.open(p)) for p in df["path"].values[idx["test"]]])
photo_scores = gate_score(embeddings(photos_heldout))   # the model only ever sees the grayscale version
eval_sets = [
    # name, colour fractions (None = already grayscale, so the colour check can't help), gate scores
    ("Test ultrasounds (should pass)", test_colour, gate_score(onnx_out["test"][1])),
    ("Colour photos, unseen categories", np.array([colour_fraction(im) for im in photos_heldout]), photo_scores),
    ("Grayscale photos, unseen categories", None, photo_scores),
    ("Chest X-rays, held out", np.array([colour_fraction(im) for im in xray_heldout]), gate_score(embeddings(xray_heldout))),
    ("Brain MRI, never seen", np.array([colour_fraction(im) for im in mri]), gate_score(embeddings(mri))),
]
ood_rows, gate_scores = [], {}
for name, colour, score in eval_sets:
    if len(score) == 0:
        print(f"{name}: dataset not attached, skipped")
        continue
    by_colour = colour > COLOUR_LIMIT if colour is not None else np.zeros(len(score), bool)
    by_gate = score < GATE_THRESHOLD
    gate_scores[name] = score
    ood_rows.append({"images": name, "n": len(score), "rejected_colour": round(float(by_colour.mean()), 4),
                     "rejected_gate": round(float(by_gate.mean()), 4),
                     "rejected_total": round(float((by_colour | by_gate).mean()), 4)})
ood_table = pd.DataFrame(ood_rows)
print(f"gate threshold {GATE_THRESHOLD:.3f}")
display(ood_table)

fig, ax = plt.subplots(figsize=(9, 4))
for name, score in gate_scores.items():
    ax.hist(score, bins=40, range=(0, 1), alpha=0.5, label=name, density=True)
ax.axvline(GATE_THRESHOLD, color="k", ls="--", label="threshold")
ax.set(xlabel="gate probability of 'breast ultrasound'", title="Ultrasound gate", yscale="log"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/breast_gate.png", dpi=130); plt.show()

## 12. Export for the Femora backend

In [ ]:
scaler, logreg = gate.named_steps["standardscaler"], gate.named_steps["logisticregression"]
np.savez(f"{OUT}/breast_gate.npz", mean=scaler.mean_.astype(np.float32), scale=scaler.scale_.astype(np.float32),
         coef=logreg.coef_[0].astype(np.float32), intercept=np.float32(logreg.intercept_[0]))

# Demo scans for the app: held-out BrEaST test images (CC BY 4.0) the model classifies correctly, median confidence
os.makedirs(f"{OUT}/samples", exist_ok=True)
test_df = df.iloc[idx["test"]].assign(pred=pred_test, conf=p_test.max(1))
samples = {}
for label in ["benign", "malignant"]:
    ok_rows = test_df[(test_df.source == "BrEaST") & (test_df.label == label) & (test_df.pred == CLASSES.index(label))]
    if len(ok_rows):
        r = ok_rows.sort_values("conf").iloc[len(ok_rows) // 2]
        shutil.copy(r["path"], f"{OUT}/samples/breast_sample_{label}.png")
        samples[label] = os.path.basename(r["path"])

df.assign(file=df["path"].map(os.path.basename))[["file", "source", "label", "group", "split"]] \
  .to_csv(f"{OUT}/breast_split.csv", index=False)

def to_builtin(o):
    return o.item() if hasattr(o, "item") else str(o)

with open(f"{OUT}/breast_meta.json", "w") as f:
    json.dump({
        "classes": CLASSES,
        "onnx_file": "breast_resnet50.onnx",
        "outputs": ["logits", "embedding", "cam"],
        "input": {"size": IMG, "mean": MEAN.ravel().tolist(), "std": STD.ravel().tolist(),
                  "preprocessing": "EXIF-rotate, grayscale, pad to square with black, bilinear resize to 224, "
                                   "repeat to 3 channels, ImageNet normalisation"},
        "temperature": TEMPERATURE,
        "malignant_threshold": MAL_THRESHOLD,
        "colour_limit": COLOUR_LIMIT,
        "gate_file": "breast_gate.npz",
        "gate_threshold": GATE_THRESHOLD,
        "recipe": {"name": BEST_RECIPE, **RECIPES[BEST_RECIPE]},
        "recipe_comparison": recipe_table.to_dict(orient="records"),
        "test_metrics": test_metrics,
        "test_metrics_argmax": argmax_metrics,
        "per_source_test": per_source,
        "comparison": comparison.to_dict(orient="records"),
        "cv_results": cv_table.to_dict(orient="records"),
        "cv_summary": cv_summary,
        "calibration": calibration,
        "cam_evaluation": cam_eval,
        "ood_evaluation": ood_table.to_dict(orient="records"),
        "onnx_parity": {"fp32_export_max_abs_diff": parity_fp32, "deployed_max_abs_diff": parity,
                        "test_predictions_changed_by_fp16": flipped},
        "best_epoch": best_epoch,
        "samples": samples,
        "dataset": {
            "sources": ["BUSI (Al-Dhabyani et al., 2020)", "BrEaST-Lesions-USG (Pawłowska et al., 2024, TCIA, CC BY 4.0)"],
            "images": int(len(df)), "near_duplicates": dedup_stats,
            "counts": pd.crosstab(df["source"], df["label"]).to_dict(),
            "splits": df["split"].value_counts().to_dict(),
        },
    }, f, indent=2, default=to_builtin)
print(sorted(os.listdir(OUT)))